In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from scipy.stats import unitary_group

In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Quantum: Nearest neighbor 2-body interactions and 1-body terms.
    Classical: ALL-TO-ALL 2-body interactions and 1-body terms.
    """
    paulis = []
    
    if model == "quantum":
        base_ops = [X, Y, Z]
        for op in base_ops: 
            for i in range(n - 1):
                j = i + 1
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
                
    elif model == "classical":
        base_ops = [Z]
        for op in base_ops: 
            for i, j in itertools.combinations(range(n), 2):
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
        
    # 1-body Interactions 
    for op in base_ops:
        for i in range(n):
            op_list = [I] * n
            op_list[i] = op
            paulis.append(krons(op_list))
    # paulis.append(krons([I] * n))
    
    return paulis

# def make_training_states(n):
#     states = []
#     dim = 2**n
#     k0, k1 = np.array([1, 0]), np.array([0, 1])
#     kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

#     # Computational basis states
#     for bits in itertools.product([k0, k1], repeat=n):
#         states.append(to_density(krons(bits)))
    
#     # +/- basis states
#     for bits in itertools.product([kp, km], repeat=n):
#         states.append(to_density(krons(bits)))

#     # GHZ state: (|00...0> + |11...1>) / sqrt(2)
#     ghz_0 = krons([k0] * n)
#     ghz_1 = krons([k1] * n)
#     ghz = (ghz_0 + ghz_1) / np.sqrt(2)
#     states.append(to_density(ghz))
    
#     # Maximally mixed state
#     states.append(np.eye(dim, dtype=complex) / dim)
    
#     # Random mixed states
#     for _ in range(3):
#         A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
#         rho = A @ A.conj().T
#         states.append(rho / np.trace(rho))

#     for _ in range(20):
#         vec = np.random.randn(dim) + 1j * np.random.randn(dim)
#         vec /= np.linalg.norm(vec)
#         states.append(np.outer(vec, vec.conj()))

#     return np.array(states)

def make_training_states(n, num_states=1000):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    return np.array(states)

def fdd_logloss_matrix(y, T, eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    derivative = -y / (1 + np.exp(y*l/T))
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

def dfj(y, rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_logloss_matrix(y, T, eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [ ]:
def make_validation_set(n, num_states=500):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    # kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)
    # import itertools
    # for bits in itertools.product([kp, km], repeat=n):
    #     states.append(to_density(krons(bits)))
            
    return np.array(states)

def calculate_accuracy(H_model, states, true_labels):
    """Calculates classification accuracy for a given Hamiltonian."""
    energies = np.array([np.real(np.trace(H_model @ rho)) for rho in states])
    predictions = np.sign(energies)
    predictions[predictions == 0] = 1 
    return np.mean(predictions == true_labels) * 100

In [ ]:
def optimize(n=3, epochs= 2000):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    val_states = make_validation_set(n, num_states=500)
    T = 2.0
    
    # compute target Hamiltonian and function outputs
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q))
    ys = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in training_states])
    ys[ys == 0] = 1

    ys_val = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in val_states])
    ys_val[ys_val == 0] = 1

    # initialize random parameters for quantum and classical Hamiltonian
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    eta = 0.1       
    
    v_q = np.zeros(len(pauli_q))
    v_c = np.zeros(len(pauli_c))

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits ---")
    print(f"{'Epoch':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")    
    print("-" * 60)
    
    for epoch in range(epochs):
        # --- Quantum Pass ---
        H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) 
        eval_q, evec_q = np.linalg.eigh(H_q)
        l_q = 0
        for i in range(N_states):
            yi = ys[i]
            m_loss_q = evec_q @ np.diag(T * np.log(1 + np.exp(-yi * eval_q / T))) @ evec_q.T.conj()
            l_q += np.real(np.trace(m_loss_q @ training_states[i]))
        l_q /= N_states
        history_q.append(l_q)
        
        grad_q = np.zeros(len(pauli_q))
        for j in range(len(pauli_q)):
            g_j = sum(dfj(ys[i], training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
            grad_q[j] = g_j / N_states
            
        est_q -= eta * grad_q

        # --- Classical Pass ---
        H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
        eval_c, evec_c = np.linalg.eigh(H_c)

        l_c = 0.0
        for i in range(N_states):
            yi = ys[i]
            m_loss_c = evec_c @ np.diag(T * np.log(1 + np.exp(-yi * eval_c / T))) @ evec_c.T.conj()
            l_c += np.real(np.trace(m_loss_c @ training_states[i]))
        l_c /= N_states
        history_c.append(l_c)
        
        grad_c = np.zeros(len(pauli_c))
        for j in range(len(pauli_c)):
            g_j = sum(dfj(ys[i], training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
            grad_c[j] = g_j / N_states
            
        est_c -= eta * grad_c

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
            "Quantum_Accuracy_Pct": None,
            "Classical_Accuracy_Pct": None
        }

        if epoch % 20 == 0:
            q_acc = calculate_accuracy(H_q, val_states, ys_val)
            c_acc = calculate_accuracy(H_c, val_states, ys_val)

            epoch_data["Quantum_Accuracy_Pct"] = q_acc
            epoch_data["Classical_Accuracy_Pct"] = c_acc
            print(f"{epoch:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(epoch_data)


    print("\n--- Final Results ---")
    final_H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
    final_H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_cl_heisenberg.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {calculate_accuracy(final_H_q, val_states, ys_val):.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_accuracy(final_H_c, val_states, ys_val):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_{\text{FCIM}}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_{\text{Heis}}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Logistic Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    
    plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    plt.title(f'{n} Qubits, Heisenberg Model', fontsize=24)
    plt.subplots_adjust(left=0.13, right=0.97, bottom=0.18, top=0.90)
    # plt.yscale('log')
    plt.legend(fontsize=24)
    plt.savefig(f"plots/logloss_{n}qubit_heis_fcim.pdf", format="pdf")
    plt.show()

In [ ]:
n = 6
history_q, history_c = optimize(n, epochs=750);

In [ ]:
plot(history_q, history_c, n)

In [ ]:
# to read data from csv instead of re-generating
n = 6
csv_filename = f"outputs/logloss_{n}qubit_heis_fcim.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)